In [11]:
import pickle
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from scipy import sparse
from scipy.sparse import diags
from sklearn.preprocessing import MinMaxScaler
from lightfm.data import Dataset
from lightfm.cross_validation import random_train_test_split
from lightfm import LightFM
from lightfm.evaluation import precision_at_k
from sklearn.feature_extraction.text import TfidfVectorizer
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score, reciprocal_rank

# Data preprocessing

## Anime Data

In [2]:
anime_df = pd.read_csv('clear_anime.csv')
anime_df['Score'] = pd.to_numeric(anime_df['Score'], errors='coerce')
anime_df['Score'] = pd.cut(
    anime_df['Score'], 
    bins=[0, 6, 7.5, 9, 10], 
    labels=['Low', 'Avg', 'High', 'Masterpiece']
)
anime_df['Year'] = pd.cut(
    anime_df['Year'], 
    bins=[0, 1990, 2000, 2010, 2020, 2023], 
    labels=['Retro', '90s', '2000s', '2010s', 'NewGen']
)
anime_df['Episodes'] = pd.cut(
    anime_df['Episodes'], 
    bins=[0, 1, 14, 27, 100, 10000], 
    labels=['Movie', 'Short', 'Medium', 'Long', 'Endless']
)
anime_df=anime_df.rename(columns={'MAL_ID': 'anime_id'})

In [3]:
anime_df

,anime_id,Score,Genres,Type,Episodes,Studios,Source,Rating,Year
0,1,High,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",TV,Medium,Sunrise,Original,R - 17+ (violence & profanity),90s
1,5,High,"Action, Drama, Mystery, Sci-Fi, Space",Movie,Movie,Bones,Original,R - 17+ (violence & profanity),2000s
2,6,High,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen",TV,Medium,Madhouse,Manga,PG-13 - Teens 13 or older,90s
3,7,Avg,"Action, Mystery, Police, Supernatural, Drama, ...",TV,Medium,Sunrise,Original,PG-13 - Teens 13 or older,2000s
4,8,Avg,"Adventure, Fantasy, Shounen, Supernatural",TV,Long,Toei Animation,Manga,PG - Children,2000s
...,...,...,...,...,...,...,...,...,...
17458,48481,Avg,"Adventure, Mystery, Supernatural",ONA,Short,Unknown,Novel,PG-13 - Teens 13 or older,NewGen
17459,48483,Avg,"Comedy, Horror, Supernatural",TV,Medium,Passione,Manga,PG-13 - Teens 13 or older,NewGen
17460,48488,Avg,"Mystery, Dementia, Horror, Psychological, Supe...",TV,Medium,Unknown,Visual novel,R - 17+ (violence & profanity),NewGen
17461,48491,Avg,"Adventure, Slice of Life, Comedy",TV,Medium,8bit,Manga,PG-13 - Teens 13 or older,2000s


In [4]:
def create_tags(row):
    tags = []
    tags.extend([f"Genre:{x.strip()}" for x in str(row['Genres']).split(',')])
    tags.append(f"Type:{str(row['Type'])}")
    tags.append(f"Score:{row['Score']}")
    tags.extend([f"Studio:{x.strip()}" for x in str(row['Studios']).split(',')])
    tags.append(f"Year:{row['Year']}")
    tags.append(f"Episodes:{row['Episodes']}")
    tags.append(f"Rating:{str(row['Rating'])}")
    tags.append(f"Source:{str(row['Source'])}")
    return tags

anime_df['feature_tags'] = anime_df.apply(create_tags, axis=1)
anime_df['feature_tags'][0]

['Genre:Action',
 'Genre:Adventure',
 'Genre:Comedy',
 'Genre:Drama',
 'Genre:Sci-Fi',
 'Genre:Space',
 'Type:TV',
 'Score:High',
 'Studio:Sunrise',
 'Year:90s',
 'Episodes:Medium',
 'Rating:R - 17+ (violence & profanity)',
 'Source:Original']

## Ratings Data

In [5]:
df = pd.read_csv('animelist.csv') 
user_counts = df['user_id'].value_counts()
active_users = user_counts[user_counts >= 50].index
df = df[df['user_id'].isin(active_users)]

valid_anime_ids = anime_df['anime_id'].unique()
df = df[df['anime_id'].isin(valid_anime_ids)]

relevant_anime_ids = df['anime_id'].unique()
anime_df = anime_df[anime_df['anime_id'].isin(relevant_anime_ids)]

In [6]:
df

,user_id,anime_id,rating
35,1,22535,9
36,1,32281,10
37,1,38000,9
38,1,18679,6
39,1,37497,8
...,...,...,...
57633243,353403,32281,10
57633244,353403,16782,10
57633245,353403,28623,8
57633246,353403,38691,9


# Dataset Preparation

In [ ]:
dataset = Dataset()

unique_feature_tags = set(x for sublist in anime_df['feature_tags'] for x in sublist)

dataset.fit(
    users=df['user_id'].unique(),
    items=anime_df['anime_id'].unique(),
    item_features=unique_feature_tags
)

(interactions, weights) = dataset.build_interactions(
    zip(df['user_id'], df['anime_id'], df['rating'])
)

item_features = dataset.build_item_features(
    zip(anime_df['anime_id'], anime_df['feature_tags'])
)

print(f"interactions shape: {interactions.shape}")
print(f"item_features shape: {item_features.shape}")

interactions shape: (223565, 16814)
item_features shape: (16814, 17594)


# Model Training

In [8]:
train, test = random_train_test_split(interactions, test_percentage=0.2, random_state=42)
train_weights, _ = random_train_test_split(weights, test_percentage=0.2, random_state=42)

In [10]:
model = LightFM(
    loss='warp',                
    no_components=64,
    learning_rate=0.05,
    item_alpha=1e-6,            
    user_alpha=1e-6,
    random_state=42
)

model.fit(
    train,
    item_features=item_features,
    sample_weight=train_weights, 
    epochs=40,
    num_threads=10,
    verbose=True
)

Epoch: 100%|██████████| 40/40 [1:43:57<00:00, 155.94s/it]


# Evaluation Results

In [16]:
def sample_matrices_for_evaluation(train, test, sample_size=5000, random_state=42):
    if not isinstance(test, sp.csr_matrix):
        test = test.tocsr()
    if not isinstance(train, sp.csr_matrix):
        train = train.tocsr()

    n_users = test.shape[0]
    test_counts = np.diff(test.indptr)
    active_users = np.where(test_counts > 0)[0]
    
    if len(active_users) > sample_size:
        rng = np.random.default_rng(random_state)
        chosen_users = rng.choice(active_users, size=sample_size, replace=False)
    else:
        chosen_users = active_users

    mask = np.zeros(n_users)
    mask[chosen_users] = 1
    diag_mask = diags(mask) 
    
    test_sampled = diag_mask.dot(test).tocsr()
    train_sampled = diag_mask.dot(train).tocsr()

    return train_sampled, test_sampled

In [18]:
train_small, test_small = sample_matrices_for_evaluation(train, test, sample_size=1000)

precision = precision_at_k(model, test_small, train_interactions=train_small,item_features=item_features,  k=10, num_threads=1).mean()
recall = recall_at_k(model, test_small, train_interactions=train_small, item_features=item_features, k=10, num_threads=1).mean()
mrr = reciprocal_rank(model, test_small, train_interactions=train_small, item_features=item_features, num_threads=1).mean()
auc = auc_score(model, test_small, train_interactions=train_small, item_features=item_features, num_threads=1).mean()

print(f"Precision@10:  {precision:.4f}")
print(f"Recall@10:     {recall:.4f}")
print(f"MRR:           {mrr:.4f}")
print(f"AUC Score:     {auc:.4f}")

Precision@10:  0.4815
Recall@10:     0.1381
MRR:           0.7869
AUC Score:     0.9832


# Model Saving

In [ ]:
data_to_save = {
    'model': model,
    'dataset': dataset,
    'item_features': item_features,
    'anime_df': anime_df 
}

filename = 'anime_recsys_model1.pkl'

with open(filename, 'wb') as f:
    pickle.dump(data_to_save, f)